# TAHMO Solar Radiation — Google Colab + Drive

**Author:** Venon Takunda Nyadombo

### Upload to Google Drive (required once)

1. On your PC, use the folder **`tahmo-solar-radiation-gdrive`** from Downloads (or zip the project with `data/Train.csv` inside).
2. In [Google Drive](https://drive.google.com): **New → Folder upload** → select that folder.
3. After upload, rename it to **`tahmo-solar-radiation`** (right-click → Rename), **or** leave the name as-is — the next cell will search for `Train.csv` automatically.

Expected layout on Drive:

```
My Drive/tahmo-solar-radiation/
  data/Train.csv
  data/Test.csv
  data/SampleSubmission.csv
  src/train_predict.py
  ...
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

DRIVE = Path('/content/drive/MyDrive')

# Add your folder name here if auto-detect fails (e.g. 'tahmo-solar-radiation-gdrive')
MANUAL_PROJECT_NAME = None  # or 'tahmo-solar-radiation-gdrive'


def find_project_dir() -> Path:
    candidates = []
    if MANUAL_PROJECT_NAME:
        candidates.append(DRIVE / MANUAL_PROJECT_NAME)
    candidates.extend([
        DRIVE / 'tahmo-solar-radiation',
        DRIVE / 'tahmo-solar-radiation-gdrive',
        DRIVE / 'Colab Notebooks' / 'tahmo-solar-radiation',
    ])
    for root in DRIVE.iterdir():
        if root.is_dir() and 'tahmo' in root.name.lower():
            candidates.append(root)

    seen = set()
    for base in candidates:
        if base in seen or not base.exists():
            continue
        seen.add(base)
        if (base / 'data' / 'Train.csv').exists():
            return base
        if (base / 'Train.csv').exists():
            return base

    # Deep search (max depth 4) for Train.csv under My Drive
    for path in DRIVE.rglob('Train.csv'):
        if 'data' in path.parts:
            return path.parent.parent
        parent = path.parent
        if (parent / 'src' / 'train_predict.py').exists():
            return parent

    print('Top-level folders on My Drive:')
    for p in sorted(DRIVE.iterdir())[:30]:
        print(' ', p.name, '(dir)' if p.is_dir() else '')
    raise FileNotFoundError(
        'Could not find data/Train.csv on Google Drive.\n'
        'Upload the project folder so you have: MyDrive/<folder>/data/Train.csv\n'
        'Then set MANUAL_PROJECT_NAME = "<folder>" above and re-run this cell.'
    )


PROJECT_DIR = find_project_dir()
os.chdir(PROJECT_DIR)
print('Working directory:', PROJECT_DIR)
print('Train.csv:', (PROJECT_DIR / 'data' / 'Train.csv').exists())
print('Files in data/:', sorted(p.name for p in (PROJECT_DIR / 'data').glob('*')))

In [ ]:
!pip install -q lightgbm pandas scikit-learn numpy

In [ ]:
!python src/train_predict.py --data-dir data --cv

In [ ]:
!python src/train_predict.py --data-dir data --blend 0.55

In [ ]:
from pathlib import Path
out = Path('output/submission.csv')
print('Submission saved to Drive:', PROJECT_DIR / out)
print('Rows:', sum(1 for _ in open(out)) - 1)